# Catalog Search — Interactive Exploration

Use this notebook to query the model, inspect scores, and compare semantic vs BM25 results.

**Pre-requisite:** run `python scripts/build_index.py` from the repo root at least once to create `indexes/`.

In [ ]:
import sys, pathlib

# Make sure the package is importable when running from notebooks/
repo_root = pathlib.Path().resolve().parent
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

INDEX_DIR = repo_root / "indexes"
print("Repo root:", repo_root)
print("Index dir:", INDEX_DIR, "— exists:", INDEX_DIR.exists())

## 1 — Load the search engine

In [ ]:
from catalog_search import CatalogSearch

# Load from the pre-built index (fast — no re-embedding)
engine = CatalogSearch.from_index(INDEX_DIR)
print(f"Model : {engine.model_name}")
print(f"Corpus: {engine.catalog_size} datasets")

## 2 — Simple search

In [ ]:
def show(results):
    """Pretty-print a list of SearchResults."""
    for r in results:
        conf = "HIGH" if r.is_high_confidence else "low "
        print(f"  [{r.rank}] {conf} {r.score:.3f}  [{r.method:8s}]  {r.dataset.id}: {r.dataset.title}")

QUERY = "longitudinal Alzheimer's imaging datasets"   # ← edit me

results = engine.search(QUERY, top_k=5)
print(f"Query: {QUERY!r}\n")
show(results)

## 3 — Compare semantic vs BM25 side-by-side

In [ ]:
QUERY = "memory loss cognitive decline"   # ← try a synonym / paraphrase

sem = engine.search(QUERY, top_k=5, force_method="semantic")
bm  = engine.search(QUERY, top_k=5, force_method="bm25")

print(f"{'SEMANTIC':^55}  {'BM25':^55}")
print("-" * 112)
for s, b in zip(sem, bm):
    left  = f"[{s.rank}] {s.score:.3f}  {s.dataset.id}: {s.dataset.title[:35]}"
    right = f"[{b.rank}] {b.score:.3f}  {b.dataset.id}: {b.dataset.title[:35]}"
    print(f"{left:<55}  {right:<55}")

## 4 — Inspect a single result in full

In [ ]:
r = results[0]   # top result from section 2

print(f"ID         : {r.dataset.id}")
print(f"Title      : {r.dataset.title}")
print(f"Score      : {r.score:.4f}  (method={r.method}, rank={r.rank})")
print(f"Modalities : {r.dataset.modality}")
print(f"Conditions : {r.dataset.conditions}")
print(f"Tags       : {r.dataset.tags}")
print(f"Source     : {r.dataset.source}")
print(f"Study type : {r.dataset.study_type}")
print()
print("Description:")
print(r.dataset.description)

## 5 — Score distribution across the whole corpus

In [ ]:
QUERY = "MRI brain imaging"   # ← edit me

all_results = engine.search(QUERY, top_k=engine.catalog_size, force_method="semantic")
scores = [r.score for r in all_results]

# ASCII histogram — no matplotlib needed
import math

buckets = 10
lo, hi = min(scores), max(scores)
width   = (hi - lo) / buckets
counts  = [0] * buckets
for s in scores:
    i = min(int((s - lo) / width), buckets - 1)
    counts[i] += 1

print(f"Score distribution for: {QUERY!r}\n")
print(f"{'Range':>14}  {'Count':>5}  Bar")
print("-" * 45)
bar_max = max(counts)
for i, c in enumerate(counts):
    lo_b = lo + i * width
    hi_b = lo_b + width
    bar  = "█" * round(30 * c / bar_max) if bar_max else ""
    print(f"  {lo_b:.3f}–{hi_b:.3f}  {c:5d}  {bar}")

print(f"\nConfidence threshold : {engine._threshold}")
print(f"High-confidence hits : {sum(1 for s in scores if s >= engine._threshold)}/{len(scores)}")

## 6 — Batch evaluation: run all test queries at once

In [ ]:
TEST_QUERIES = [
    ("longitudinal Alzheimer's imaging",      {"ADNI-001", "OASIS-002", "DIAN-095"}),
    ("MRI brain imaging",                     {"ADNI-001", "HCP-003",   "ENIGMA-005"}),
    ("cancer genomics sequencing",            {"TCGA-007", "ICGC-058",  "BRCA-010"}),
    ("type 2 diabetes randomised trial",      {"ACCORD-014","UKPDS-023","DPPOS-024"}),
    ("memory loss cognitive decline",         {"ADNI-001", "OASIS-002", "DIAN-095"}),
    ("heart attack cardiovascular prevention",{"FHS-011",  "MESA-012",  "SPRINT-043"}),
    ("gut bacteria bowel inflammation",       {"MICROBIOME-HMP-100"}),
    ("eye retina diabetic photographs",       {"EYEPACS-037","AREDS-057"}),
]

K = 5
print(f"{'Query':<45}  P@{K}   Top-1 ID")
print("-" * 75)
hits_total = found_total = 0
for query, relevant in TEST_QUERIES:
    res = engine.search(query, top_k=K)
    top_ids = {r.dataset.id for r in res}
    precision = len(top_ids & relevant) / K
    found = bool(top_ids & relevant)
    hits_total  += len(top_ids & relevant)
    found_total += K
    flag = "✓" if found else "✗"
    print(f"  {flag} {query:<43}  {precision:.2f}   {res[0].dataset.id}")

print(f"\nOverall P@{K}: {hits_total / found_total:.2f}")

## 7 — Embed your own text and find nearest neighbours

In [ ]:
from catalog_search.embedder import BiomedEmbedder
import numpy as np

embedder = BiomedEmbedder()   # reuses cached model weights

texts = [
    "longitudinal MRI of Alzheimer patients",
    "brain scan memory loss study",          # paraphrase
    "heart attack prevention cohort",        # different domain
]

vecs = embedder.encode(texts)

# Pairwise cosine similarities (vectors are already L2-normalised)
print("Pairwise cosine similarities:\n")
for i, t1 in enumerate(texts):
    for j, t2 in enumerate(texts):
        if j > i:
            sim = float(np.dot(vecs[i], vecs[j]))
            print(f"  {sim:.4f}  \"{t1}\"  ↔  \"{t2}\"")